In [1]:
# Import libraries for data preprocessing
import numpy as np
import pandas as pd

# Import libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import  LinearRegression
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.distributions import Normal
from sklearn.metrics import mean_absolute_error
from torch.distributions import StudentT

#### Data Preprocessing

In [3]:
# Loading dataset
df = pd.read_csv("mcs_ds_edited_iter_shuffled.csv")
df.head()

# Split into x and y where x is the first 4 columns and y is ale
x = df.iloc[:, :4]
y = df['ale']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

(85, 4)
(22, 4)
(85,)
(22,)


#### Baseline

In [7]:
# Linear Regression
baseline = LinearRegression()

# Train the model
baseline.fit(X_train, y_train)

# Predict y-values and calculate residuals
baseline_pred = baseline.predict(X_test)

# Calculate accuracy score
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline linear regression MAE: {baseline_mae:.4f}") 

Baseline linear regression MAE: 0.1954


#### Linear Regression

In [ ]:
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(input_dim, 1) * 0.01)
        self.b = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        return x @ self.w + self.b

# Custom MSE Loss
def custom_mse_loss(y_pred, y_true):
    return torch.mean((y_true - y_pred) ** 2)

# Train Linear Regression
mse_model = LinearRegressionModel(X_train_t.shape[1])
optimizer_mse = SGD(mse_model.parameters(), lr=1e-6)

for epoch in range(20000):
    optimizer_mse.zero_grad()
    predictions = mse_model(X_train_t)
    loss = custom_mse_loss(predictions, y_train_t)
    loss.backward()
    optimizer_mse.step()

In [ ]:
# Evaluate Linear Regression
with torch.no_grad():
    mse_test_preds = mse_model(X_test_t)
    mse_mae = torch.mean(torch.abs(y_test_t - mse_test_preds)).item()

print(f"Scikit-Learn Baseline MAE : {baseline_mae:.4f}")
print(f"Custom PyTorch MSE MAE     : {mse_mae:.4f}")

Scikit-Learn Baseline MAE : 0.1954
Custom PyTorch MSE MAE     : 0.3496


The performance seems worse than baseline since we are supposed to be minizing MAE

#### Linear Regression using T distribution

In [24]:
def t_distribution_loss(y_pred, y_true, df=5.0):    
    # T distribution with 5 degrees of freedom
    t_dist = StudentT(df=df, loc=0.0, scale=1.0)
    
    # Minimize the Negative Log-Likelihood (NLL)
    return -torch.mean(t_dist.log_prob(y_true - y_pred))

In [25]:
# Train Linear Regression with T-distribution loss
t_model = LinearRegressionModel(X_train_t.shape[1])
optimizer_t = SGD(t_model.parameters(), lr=1e-6)

for epoch in range(20000):
    optimizer_t.zero_grad()
    predictions = t_model(X_train_t)
    loss = t_distribution_loss(predictions, y_train_t, df=5.0)
    loss.backward()
    optimizer_t.step()

In [26]:
with torch.no_grad():
    t_test_preds = t_model(X_test_t)
    t_mae = torch.mean(torch.abs(y_test_t - t_test_preds)).item()

print(f"Scikit-Learn Baseline MAE : {baseline_mae:.4f}")
print(f"Custom PyTorch Student-T MAE: {t_mae:.4f}")

Scikit-Learn Baseline MAE : 0.1954
Custom PyTorch Student-T MAE: 0.3721


The performance seems worse than baseline since we are supposed to be minizing MAE